# WiSenseHub Quickstart

This notebook mirrors the GitHub Pages tutorial. It starts with the synthetic fixture so the pipeline can run without downloading restricted research data, then shows the command pattern for real official releases.

## 1. Install the project

Run this from the repository root. If you already installed the package, skip this cell.

In [ ]:
# !pip install -e ".[dev]"

## 2. Create and standardize a tiny synthetic CSI file

In [ ]:
!python examples/make_synthetic_input.py
!python -m wifi_datahub standardize \
  --input examples/data/synthetic_csi.csv \
  --output examples/data/standardized/synthetic_csi.npz \
  --dataset-id synthetic-demo \
  --sample-rate 100 \
  --duration 4

## 3. Inspect the NumPy output contract

In [ ]:
import json
from pathlib import Path
import numpy as np

path = Path("examples/data/standardized/synthetic_csi.npz")
sample = np.load(path)
for name in sample.files:
    arr = sample[name]
    print(f"{name:14s} shape={arr.shape} dtype={arr.dtype}")

The standard output is framework-neutral NumPy. Native CSI-like arrays usually use time, link/channel, and subcarrier axes. A derived view can later flatten or fix these axes for training.

In [ ]:
t = sample["timestamp_s"]
amp = sample["amplitude"]
trace = amp[:, 0, 0]
print("first timestamps:", np.round(t[:8], 3))
print("first amplitude values:", np.round(trace[:8], 3))
print("mean/std:", float(trace.mean()), float(trace.std()))

## 4. Generate a quality report

In [ ]:
!python -m wifi_datahub quality \
  --input examples/data/standardized/synthetic_csi.npz \
  --output examples/data/standardized/synthetic_csi.quality.json

with open("examples/data/standardized/synthetic_csi.quality.json") as f:
    report = json.load(f)
report["arrays"]["amplitude"]

## 5. Switch to a real official release

For a real dataset, download the original files from the official source and keep the directory tree intact under `data/<dataset-id>/original/`.

```bash
python -m wifi_datahub settings ut-har
python -m wifi_datahub prepare ut-har \
  --setting official \
  --data-root data \
  --target-length 128 \
  --layout link-subcarrier \
  --interpolation linear
```

Use `python -m wifi_datahub settings <dataset-id>` to see supported split settings such as official, random, cross-subject, or cross-device.